Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # main.py

 **Entry point** — wires the lifespan, middleware, and routes together.

 ## Full project flow

 ```
 main.py (lifespan)
   → init_database()       database.py
   → init_agent()          graph.py → short_term.py

 HTTP request → routes/agent.py
   → LTM.load()            long_term.py
   → graph.ainvoke()       graph.py
       → input_guard → planner → agent ↔ tools → output_guard
                                     ↓
                                human_loop (if needed)
   → LTM.extract() + update()
 ```

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager

from app.core.config import get_settings
from app.core.database import init_database, close_db
from app.core.logging import get_logger
from app.routes.agent import router as api_router
from app.agent.graph import init_agent
from app.agent.memory.short_term import close_checkpointer

settings = get_settings()
logger = get_logger(__name__)

 ## Lifespan

 Everything before `yield` runs at startup, everything after at shutdown.
 Replaces the old `@app.on_event("startup")` pattern — startup and shutdown
 are co-located in one context manager.

 **Startup order is required:**
 1. `init_database()` — SQLAlchemy engine + pool + `agent_memory` table
 2. `init_agent()` — psycopg3 pool + checkpointer + graph compilation

 Swap them and it crashes — the graph needs the checkpointer,
 the checkpointer needs the database to exist.

 **Bug fix:** `close_db()` was called inside the `if` branch at startup
 when `DATABASE_URL` is missing. Nothing is open at that point, so it was
 harmless, but logically wrong. Moved to shutdown only.

 **CORS:** hardcoded to `localhost:3000` for development.
 In production, pull from `settings.cors_origins` (already defined in `config.py`).

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup
    if settings.database_url:
        await init_database()
        await init_agent()
        logger.info("agent_ready")
    else:
        logger.warning("DATABASE_URL not configured — agent disabled")

    yield

    # Shutdown
    await close_checkpointer()
    await close_db()

In [ ]:
app = FastAPI(
    title=settings.app_name,
    debug=settings.debug,
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=settings.cors_origins,  # fix: use config instead of hardcoded value
    allow_credentials=True,
    allow_headers=["*"],
    allow_methods=["*"],
)

app.include_router(api_router)

 ## Endpoints système

In [ ]:
@app.get("/", tags=["Root"])
async def root() -> dict:
    return {
        "message": "React Agent API is running.",
        "health":  "/health",
        "docs":    "/docs",
        "agent":   "/agent/chat",
    }


@app.get("/health", tags=["Health"])
async def health_check() -> dict:
    """Returns environment name — confirms whether you're hitting dev or prod."""
    return {"status": "ok", "environment": settings.environment}





# if __name__ == "__main__":
#     import uvicorn
#     uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=False)